# NF2 SHARP CEA Extrapolation In Google Colab

This notebook is the Colab-friendly SHARP CEA workflow: install NF2 from GitHub, download one vector magnetogram, train a compact Cartesian extrapolation, export the result, and inspect a few diagnostics.

The defaults favor a quick run on a single Colab GPU. Increase `MODEL_DIM`, `ITERATIONS`, or `EPOCHS` for more complete production runs.

## 1. Install NF2

Colab already provides Python and usually a CUDA-enabled PyTorch build. This cell installs NF2 and the JSOC/W&B helper dependencies used by the notebook. If Colab asks you to restart the runtime after installation, restart and then continue with the next cell.

In [ ]:
%pip install nf2


## 2. Configure The Run

`RUN_DOWNLOAD` and `RUN_TRAINING` default to `True` so the notebook performs a complete example. The default `MODEL_FIELD = "b"` trains the magnetic field directly and adds a divergence loss with `DIVERGENCE_WEIGHT`. Set `MODEL_FIELD = "vector_potential"` to train a vector potential model with `B = curl(A)`.

`FORCE_FREE_WEIGHT` is the primary tuning parameter for this example. Increase it to push the extrapolation toward a more force-free solution, especially when the final `theta_J_deg` metric is greater than about `20` degrees. Decrease it when the model boundary becomes too smooth or no longer matches the observed boundary closely enough. In practice this is a tradeoff between volume force-freeness and boundary fidelity.

The SHARP download requests both field components and uncertainty maps. The file matching below is exact, so `Br_err.fits` cannot be confused with `Br.fits`.

In [ ]:
from pathlib import Path

RUN_DOWNLOAD = True
RUN_TRAINING = True

JSOC_EMAIL = "you@example.org"
SHARP_NUM = 377
NOAA_NUM = None
T_START = "2011-02-15T00:00:00"
T_END = "2011-02-15T00:12:00"
CADENCE = "720s"
SERIES = "sharp_cea_720s"
SEGMENTS = "Br,Bp,Bt,Br_err,Bp_err,Bt_err"

MODEL_FIELD = "b"  # "b" or "vector_potential"
MODEL_DIM = 128  # 256 for default setting, 128 for faster extrapolations
FORCE_FREE_WEIGHT = 1.0e-3  # primary tuning parameter: larger = more force-free, smaller = closer boundary fit
DIVERGENCE_WEIGHT = 1.0e-4

EPOCHS = 15
ITERATIONS = 10000
SAMPLER_BATCH_SIZE = 8192
BOUNDARY_BATCH_SIZE = 4096
VALIDATION_BATCH_SIZE = 4096
VALIDATION_PIXEL_PER_DS = 64 # reduce for faster validation, increase for better validation metric estimates
Z_RANGE = [0, 80] # adjust height to match the horizontal extent of the extrapolation domain and the expected height of the field structures

WANDB_MODE = "online"

RUN_DIR = Path("/content/runs/sharp_cea") if Path("/content").exists() else Path("runs/colab_sharp_cea")
DATA_DIR = RUN_DIR / "data"
WORK_DIR = RUN_DIR / "work"
NF2_PATH = RUN_DIR / "extrapolation_result.nf2"
EXPORT_DIR = RUN_DIR / "exports"

BR_FITS = DATA_DIR / "Br.fits"
BT_FITS = DATA_DIR / "Bt.fits"
BP_FITS = DATA_DIR / "Bp.fits"
BR_ERR_FITS = DATA_DIR / "Br_err.fits"
BT_ERR_FITS = DATA_DIR / "Bt_err.fits"
BP_ERR_FITS = DATA_DIR / "Bp_err.fits"

EXPORT_MM_PER_PIXEL = 0.72
HEIGHT_RANGE = [0, 80]


## 3. Import Packages And Check The Runtime

Use a GPU runtime in Colab for training: `Runtime > Change runtime type > T4 GPU` or another available GPU. The CUDA printout should report at least one GPU.

In [ ]:
import os
os.environ["WANDB_MODE"] = WANDB_MODE
os.environ.setdefault("WANDB_CONSOLE", "off")

from dateutil.parser import parse
from astropy import units as u
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import numpy as np
import torch
import wandb

import nf2
from nf2.evaluation.metric import normalized_divergence, sigma_J, theta_J, vector_norm

print("NF2:", nf2.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
for idx in range(torch.cuda.device_count()):
    print(f"CUDA device {idx}:", torch.cuda.get_device_name(idx))


## 4. Download SHARP CEA Files

This cell downloads the requested SHARP CEA segments from JSOC when `RUN_DOWNLOAD = True`. If you already uploaded FITS files to `DATA_DIR`, set `RUN_DOWNLOAD = False` and keep the same file naming or let the exact segment matcher find JSOC-style filenames.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
if RUN_DOWNLOAD:
    nf2.download_sharp_series(
        str(DATA_DIR), JSOC_EMAIL, parse(T_START), parse(T_END),
        noaa_num=NOAA_NUM, sharp_num=SHARP_NUM, cadence=CADENCE,
        segments=SEGMENTS, series=SERIES,
    )

def find_segment(segment, fallback):
    matches = sorted(DATA_DIR.glob(f"*.{segment}.fits"))
    return matches[0] if matches else fallback

BR_FITS = find_segment("Br", BR_FITS)
BT_FITS = find_segment("Bt", BT_FITS)
BP_FITS = find_segment("Bp", BP_FITS)
BR_ERR_FITS = find_segment("Br_err", BR_ERR_FITS)
BT_ERR_FITS = find_segment("Bt_err", BT_ERR_FITS)
BP_ERR_FITS = find_segment("Bp_err", BP_ERR_FITS)

required_files = [BR_FITS, BT_FITS, BP_FITS, BR_ERR_FITS, BT_ERR_FITS, BP_ERR_FITS]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required FITS files:\n" + "\n".join(missing))

print("Field files:")
print(BR_FITS, BT_FITS, BP_FITS, sep="\n")
print("Error files:")
print(BR_ERR_FITS, BT_ERR_FITS, BP_ERR_FITS, sep="\n")


## 5. Build The NF2 Configuration

The configuration uses the observed SHARP CEA boundary plus an explicit potential side/top boundary. The default `MODEL_FIELD = "b"` also adds divergence control, while the potential-boundary and potential losses keep the outer volume anchored during the short Colab run.

In [ ]:
if MODEL_FIELD not in {"vector_potential", "b"}:
    raise ValueError('MODEL_FIELD must be "vector_potential" or "b".')

losses = [
    {"type": "boundary", "name": "boundary", "weight": 1.0, "datasets": "boundary"},
    {"type": "boundary", "name": "potential_boundary", "weight": 10.0, "datasets": "potential"},
    {"type": "force_free", "name": "force_free", "weight": FORCE_FREE_WEIGHT, "datasets": ["random"]},
]
if MODEL_FIELD == "b":
    losses.append({"type": "divergence", "name": "divergence", "weight": DIVERGENCE_WEIGHT, "datasets": ["random"]})

loss_scaling = [{"type": "b_height", "name": "b_height", "loss_ids": [loss["name"] for loss in losses if loss["type"] != "boundary"]}]

config = {
    "path": str(RUN_DIR),
    "work_path": str(WORK_DIR),
    "logging": {"project": "nf2", "name": f"SHARP CEA Colab {MODEL_FIELD}"},
    "model": {"field": MODEL_FIELD, "network": {"hidden_dim": MODEL_DIM}},
    "data": {
        "geometry": "cartesian",
        "boundaries": [{
            "id": "boundary",
            "type": "sharp",
            "files": {"Br": str(BR_FITS), "Bt": str(BT_FITS), "Bp": str(BP_FITS)},
            "errors": {"Br_err": str(BR_ERR_FITS), "Bt_err": str(BT_ERR_FITS), "Bp_err": str(BP_ERR_FITS)},
            "batch_size": BOUNDARY_BATCH_SIZE,
        }],
        "sampler": {"type": "height", "batch_size": SAMPLER_BATCH_SIZE},
        "potential_boundary": {"type": "potential", "strides": 4},
        "validation": [
            {"id": "boundary", "type": "sharp", "files": {"Br": str(BR_FITS), "Bt": str(BT_FITS), "Bp": str(BP_FITS)}, "errors": {"Br_err": str(BR_ERR_FITS), "Bt_err": str(BT_ERR_FITS), "Bp_err": str(BP_ERR_FITS)}, "batch_size": VALIDATION_BATCH_SIZE},
            {"id": "cube", "type": "cube", "batch_size": VALIDATION_BATCH_SIZE},
            {"id": "slices", "type": "slices", "n_slices": 8, "batch_size": VALIDATION_BATCH_SIZE},
        ],
        "iterations": ITERATIONS,
        "z_range": Z_RANGE,
        "validation_pixel_per_ds": VALIDATION_PIXEL_PER_DS,
    },
    "training": {"epochs": EPOCHS},
    "losses": losses,
    "loss_scaling": loss_scaling,
    "callbacks": [
        {"type": "boundary", "dataset": "boundary"},
        {"type": "metrics", "dataset": "cube"},
        {"type": "slices", "dataset": "slices"},
    ],
}

config


## 6. Train The Extrapolation

Run the training cell to launch NF2 and stream metrics/images to W&B. `WANDB_CONSOLE=off` avoids notebook console recursion while leaving W&B online logging enabled.

In [ ]:
if RUN_TRAINING:
    nf2.run(**config)
    wandb.finish()
else:
    config


## 7. Export And Evaluate

After training, export VTK/NumPy files and compute compact quality diagnostics. Download files from the Colab file browser if you want to inspect the VTK output in ParaView.

Use `theta_J_deg` as the main force-free alignment check. If `theta_J_deg` is above about `20`, rerun with a larger `FORCE_FREE_WEIGHT`. If boundary plots look too smooth or underfit the magnetogram, rerun with a smaller `FORCE_FREE_WEIGHT`.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / "field.vtk"), fmt="vtk", Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=HEIGHT_RANGE, metrics=["j", "alpha", "free_energy_fft"])
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / "field.npz"), fmt="npz", Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=HEIGHT_RANGE, metrics=["j", "alpha", "free_energy_fft"])

out = nf2.load(NF2_PATH)
cube = out.load_cube(Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=HEIGHT_RANGE, metrics=["j", "free_energy_fft"], progress=True)

b = cube["b"].to_value(u.G)
j_current = cube["metrics"]["j"].to_value(u.G / u.s)
j = vector_norm(j_current)
free_energy = cube["metrics"]["free_energy_fft"].to_value(u.erg / u.cm**3)
voxel_volume_cm3 = (EXPORT_MM_PER_PIXEL * u.Mm).to_value(u.cm) ** 3
metrics = {
    "mean_normalized_divergence": float(np.nanmean(normalized_divergence(b))),
    "theta_J_deg": float(theta_J(b, j_current)),
    "sigma_J_1e2": float(sigma_J(b, j_current) * 1e2),
    "mean_B_norm": float(np.nanmean(vector_norm(b))),
    "mean_J_norm": float(np.nanmean(j)),
    "total_free_energy_erg": float(np.nansum(free_energy) * voxel_volume_cm3),
}
metrics


## 8. Quick-Look Plots

These maps give a fast sanity check: integrated current density, integrated free-energy density, and the modeled lower-boundary `Bz`.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

current_map = np.nansum(j, axis=2) * EXPORT_MM_PER_PIXEL
free_energy_map = np.nansum(free_energy, axis=2) * (EXPORT_MM_PER_PIXEL * u.Mm).to_value(u.cm)
boundary_bz = b[:, :, 0, 2]

extent = [0, b.shape[0] * EXPORT_MM_PER_PIXEL, 0, b.shape[1] * EXPORT_MM_PER_PIXEL]

def log_norm(image, lower=1, upper=99):
    positive = image[np.isfinite(image) & (image > 0)]
    if positive.size == 0:
        return LogNorm(vmin=np.nextafter(0, 1), vmax=1.0)
    vmin, vmax = np.nanpercentile(positive, [lower, upper])
    return LogNorm(vmin=max(vmin, np.nextafter(0, 1)), vmax=max(vmax, vmin * 1.01))

im = axs[0].imshow(current_map.T, origin="lower", cmap="inferno", norm=log_norm(current_map), extent=extent)
axs[0].set_title("Height-integrated |J| (log)")
plt.colorbar(im, ax=axs[0], fraction=0.046, label="Height-integrated |J| [G Mm s$^{-1}$]")

im = axs[1].imshow(free_energy_map.T, origin="lower", cmap="jet", norm=log_norm(free_energy_map), extent=extent)
axs[1].set_title("Height-integrated free energy (log)")
plt.colorbar(im, ax=axs[1], fraction=0.046, label="Height-integrated free energy [erg cm$^{-2}$]")

lim = np.nanpercentile(np.abs(boundary_bz), 99)
im = axs[2].imshow(boundary_bz.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim, extent=extent)
axs[2].set_title("Model boundary Bz")
plt.colorbar(im, ax=axs[2], fraction=0.046, label="$B_z$ [G]")

for ax in axs:
    ax.set_xlabel("x [Mm]")
    ax.set_ylabel("y [Mm]")
fig.tight_layout()
plt.show()
